# 🎵 Máy chủ AI cho app Bài Hát Tặng (miễn phí)

**Cách dùng:**
1. Trên thanh menu chọn **Runtime → Change runtime type → T4 GPU → Save** (chỉ cần làm lần đầu).
2. Bấm **Runtime → Run all** (hoặc Ctrl+F9). Lần đầu mất khoảng 5–10 phút để cài và tải mô hình.
3. Kéo xuống **ô số 4 (Tạo LINK MÁY CHỦ)**. Khi dưới ô đó hiện khung xanh **LINK MÁY CHỦ** dạng `https://....trycloudflare.com`, chép link đó.
4. Mở app → ⚙️ Cài đặt → chọn **Máy chủ Colab** → dán link → Lưu → tạo bài hát.

⚠️ Giữ trang này mở trong lúc dùng app. Đóng trang hoặc để lâu không dùng thì Colab tự tắt; khi đó chạy lại và dán link mới.

In [ ]:
#@title 1. Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo '❌ Chưa bật GPU: Runtime → Change runtime type → T4 GPU'

In [ ]:
#@title 2. Cài ACE-Step 1.5 (mất vài phút)
import os, subprocess
%cd /content
if not os.path.exists('/content/ace'):
    !git clone -q --depth 1 https://github.com/ace-step/ACE-Step-1.5 /content/ace
%cd /content/ace
!pip -q install uv
!uv sync -q 2>&1 | tail -5
print('✅ Đã cài xong')

In [ ]:
#@title 3. Khởi động máy chủ (lần đầu tải mô hình khoảng 3–5 phút)
wrapper = r'''
import sys
import acestep.api.route_setup as rs
from fastapi.middleware.cors import CORSMiddleware as _C
class OpenCORS(_C):
    def __init__(self, app, **kw):
        super().__init__(app, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])
rs.CORSMiddleware = OpenCORS   # cho phép app điện thoại gọi tới
from acestep.api_server import main
sys.argv = ['acestep-api', '--host', '0.0.0.0', '--port', '8001']
main()
'''
open('/content/ace/run_server.py','w').write(wrapper)
import subprocess, time, urllib.request
subprocess.run('pkill -f run_server.py', shell=True)
log = open('/content/server.log','w')
srv = subprocess.Popen(['uv','run','--no-sync','python','run_server.py'], cwd='/content/ace', stdout=log, stderr=subprocess.STDOUT)
ok = False
for i in range(240):
    time.sleep(5)
    try:
        urllib.request.urlopen('http://127.0.0.1:8001/health', timeout=3); ok = True; break
    except Exception:
        if srv.poll() is not None: break
        if i % 6 == 0: print('⏳ Đang khởi động...', i*5, 'giây')
print('✅ Máy chủ đã chạy' if ok else '❌ Máy chủ lỗi, xem log dưới')
if not ok: print(open('/content/server.log').read()[-3000:])

In [ ]:
#@title 4. Tạo LINK MÁY CHỦ để dán vào app
import subprocess, re, time, os
if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
subprocess.run('pkill -f cloudflared', shell=True)
tl = open('/content/tunnel.log','w')
tun = subprocess.Popen(['/content/cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8001'], stdout=tl, stderr=subprocess.STDOUT)
url = None
for i in range(60):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m: url = m.group(0); break
from IPython.display import HTML, display
if url:
    print('\n' + '='*60)
    print('LINK MÁY CHỦ (chép dòng dưới đây):')
    print(url)
    print('='*60 + '\n')
    display(HTML(f"""<div style="font-size:18px;padding:16px;border:3px solid #0a78bd;border-radius:12px;background:#eef6fd;color:#1e3a5f">
    🔗 <b>LINK MÁY CHỦ:</b><br>
    <input id="lk" value="{url}" readonly onclick="this.select()" style="width:100%;font-size:18px;padding:10px;margin:10px 0;border:2px solid #0a78bd;border-radius:8px">
    <button onclick="navigator.clipboard.writeText('{url}').then(()=>this.innerText='✅ Đã chép! Dán vào app')" style="font-size:18px;padding:12px 18px;background:#0a78bd;color:#fff;border:0;border-radius:10px">📋 Chép link</button>
    <p>Dán vào app → ⚙️ Cài đặt → Máy chủ Colab → Lưu. Giữ trang này mở khi dùng app.</p></div>"""))
else:
    print('❌ Chưa tạo được link, bấm chạy lại ô này'); print(open('/content/tunnel.log').read()[-1500:])


In [ ]:
#@title 5. (Tùy chọn) Xem nhật ký máy chủ
!tail -n 40 /content/server.log